In [ ]:
# Force delete old price and news data files before collection
import os
import glob
for symbol in SYMBOLS:
    price_path = f"data/price_data/{symbol}_price.json"
    news_path = f"data/news_data/{symbol}_news.json"
    if os.path.exists(price_path):
        os.remove(price_path)
    if os.path.exists(news_path):
        os.remove(news_path)
print("✅ Old data files deleted. Ready for fresh collection.")

# 🛠️ Data Collection for Backtesting

This notebook collects price and news data for backtesting the FinBERT-LSTM trading system.

## ⚠️ **Important: Alpaca News API Limitations**

**The Alpaca free tier API limits news results to approximately 50 articles per symbol**, regardless of the time period requested. This is a known limitation of the free tier.

### Implications:
- For a multi-year period (2020-2025), you will only get ~50 recent news articles per symbol
- This may not be sufficient for robust backtesting over long periods
- Consider these alternatives:
  1. **Upgrade to paid Alpaca tier** for unlimited news access
  2. **Use alternative news sources** (NewsAPI, Alpha Vantage, web scraping)
  3. **Adjust backtest period** to recent months where 50 articles may be sufficient
  4. **Focus on price-based features** with limited sentiment features

### Current Status:
✅ Pagination using `page_token` implemented correctly  
✅ Timezone handling fixed (UTC)  
✅ Proper API response parsing  
⚠️ Limited to ~50 articles per symbol due to API tier

In [39]:
# OPTIONAL: Test single symbol collection before running full collection
import importlib
import sys
sys.path.append('../')

# Clear any existing imports
if 'utils.backtest_utils' in sys.modules:
    del sys.modules['utils.backtest_utils']

from utils.backtest_utils import *

# Test single symbol news collection
data_manager = BacktestDataManager()
print("🧪 Testing news collection for AAPL (short period)...")
print("This test verifies the pagination logic is working correctly.\n")

# Delete old test file
import os
test_file = 'data/news_data/AAPL_news.json'
if os.path.exists(test_file):
    os.remove(test_file)

# Test with a short period
test_start = datetime.strptime('2024-01-01', "%Y-%m-%d")
test_end = datetime.strptime('2024-01-31', "%Y-%m-%d")

data_manager._collect_news_data('AAPL', test_start, test_end)

# Check results
test_data = data_manager.load_json(test_file)
article_count = len(test_data.get('records', []))

print(f"\n{'='*60}")
print(f"✅ Test complete! Collected {article_count} articles for AAPL in Jan 2024")
print(f"{'='*60}")

if article_count == 50:
    print("\n⚠️  Got exactly 50 articles - this is likely the API limit.")
    print("The pagination code is working, but Alpaca free tier caps results.")
elif article_count > 50:
    print("\n✅ Got more than 50 articles - pagination is working!")
else:
    print(f"\nℹ️  Got {article_count} articles (less than API limit).")

🧠 Loading FinBERT...
🧪 Testing news collection for AAPL (short period)...
This test verifies the pagination logic is working correctly.

  📰 Fetching news from 2024-01-01 to 2024-01-31...
🧪 Testing news collection for AAPL (short period)...
This test verifies the pagination logic is working correctly.

  📰 Fetching news from 2024-01-01 to 2024-01-31...
    Page 1: 50 articles (total: 50)
    ℹ️ Note: Alpaca API may limit results. Got exactly 50 articles.
  📰 News: 50 articles saved

✅ Test complete! Collected 50 articles for AAPL in Jan 2024

⚠️  Got exactly 50 articles - this is likely the API limit.
The pagination code is working, but Alpaca free tier caps results.
    Page 1: 50 articles (total: 50)
    ℹ️ Note: Alpaca API may limit results. Got exactly 50 articles.
  📰 News: 50 articles saved

✅ Test complete! Collected 50 articles for AAPL in Jan 2024

⚠️  Got exactly 50 articles - this is likely the API limit.
The pagination code is working, but Alpaca free tier caps results.


In [40]:
# Cell 1: Setup and Configuration
import sys
sys.path.append('../')

# Reload the module to get latest changes
if 'utils.backtest_utils' in sys.modules:
    del sys.modules['utils.backtest_utils']

from utils.backtest_utils import *

print("📊 FinBERT-LSTM Backtesting System")
print("="*50)
print(f"📅 Data Period: {START_DATE} to {END_DATE}")
print(f"📈 Symbols: {SYMBOLS}")
print(f"🔄 Walk-Forward: {TRAIN_MONTHS}m train + {VAL_MONTHS}m val + {TEST_MONTHS}m test")
print("\n⚠️  Note: Alpaca free tier limits news to ~50 articles/symbol")
print("="*50)

# Cell 2: Initialize Data Manager and Collect All Data
data_manager = BacktestDataManager()

# Check if data already exists
data_exists = all([
    os.path.exists(f"data/price_data/{symbol}_price.json") and 
    os.path.exists(f"data/news_data/{symbol}_news.json")
    for symbol in SYMBOLS
])

if not data_exists:
    print("\n📥 Collecting fresh data...")
    print("This may take several minutes due to sentiment analysis...\n")
    data_manager.collect_all_data()
else:
    print("\n✅ Data already exists. Delete files to force recollection.")
    print("Run the first cell to delete old data if needed.\n")

# Cell 3: Verify Data Collection
print("\n🔍 DATA VERIFICATION")
print("="*60)

total_price_records = 0
total_news_articles = 0

for symbol in SYMBOLS:
    price_data = data_manager.load_json(f"data/price_data/{symbol}_price.json")
    news_data = data_manager.load_json(f"data/news_data/{symbol}_news.json")
    
    price_count = len(price_data.get('records', []))
    news_count = len(news_data.get('records', []))
    
    total_price_records += price_count
    total_news_articles += news_count
    
    print(f"{symbol:6s}: {price_count:4d} prices, {news_count:3d} news articles")

print("="*60)
print(f"TOTAL:  {total_price_records:4d} prices, {total_news_articles:3d} news articles")
print("\n✅ Data collection completed!")

# Cell 4: Data Quality Analysis
print("\n📊 DATA QUALITY ANALYSIS")
print("="*60)

for symbol in SYMBOLS:
    price_data = data_manager.load_json(f"data/price_data/{symbol}_price.json")
    news_data = data_manager.load_json(f"data/news_data/{symbol}_news.json")
    
    if not price_data.get('records'):
        print(f"❌ {symbol}: No price data")
        continue
    
    # Price data analysis
    dates = [record['date'] for record in price_data['records']]
    price_start = min(dates)
    price_end = max(dates)
    
    # News data analysis
    news_count = len(news_data.get('records', []))
    if news_count > 0:
        news_dates = [record['date'] for record in news_data['records']]
        news_start = min(news_dates)
        news_end = max(news_dates)
        
        # Calculate average sentiment
        avg_sentiment = sum(r['sentiment_score'] for r in news_data['records']) / news_count
        
        print(f"\n{symbol}:")
        print(f"  Price: {len(dates)} days ({price_start} to {price_end})")
        print(f"  News:  {news_count} articles ({news_start} to {news_end})")
        print(f"  Avg Sentiment: {avg_sentiment:+.3f} ({'Bullish' if avg_sentiment > 0 else 'Bearish'})")
        
        if news_count == 50:
            print(f"  ⚠️  Limited to 50 articles (API restriction)")
    else:
        print(f"\n{symbol}:")
        print(f"  Price: {len(dates)} days ({price_start} to {price_end})")
        print(f"  News:  ❌ No news articles found")

print("\n" + "="*60)
print("✅ Data quality check complete!")
print("\nℹ️  If news counts are low (~50), this is due to Alpaca API limits.")
print("Consider using a shorter time period or alternative news sources.")

🧠 Loading FinBERT...
📊 FinBERT-LSTM Backtesting System
📅 Data Period: 2020-01-01 to 2025-10-1
📈 Symbols: ['AAPL', 'TSLA', 'MSFT', 'GOOGL', 'NVDA']
🔄 Walk-Forward: 25m train + 5m val + 3m test

⚠️  Note: Alpaca free tier limits news to ~50 articles/symbol

✅ Data already exists. Delete files to force recollection.
Run the first cell to delete old data if needed.


🔍 DATA VERIFICATION
AAPL  : 1444 prices,  50 news articles
TSLA  : 1444 prices,   0 news articles
MSFT  : 1444 prices,   0 news articles
GOOGL : 1444 prices,   0 news articles
NVDA  : 1444 prices,   0 news articles
TOTAL:  7220 prices,  50 news articles

✅ Data collection completed!

📊 DATA QUALITY ANALYSIS

AAPL:
  Price: 1444 days (2020-01-02 to 2025-09-30)
  News:  50 articles (2024-01-26 to 2024-01-30)
  Avg Sentiment: +0.180 (Bullish)
  ⚠️  Limited to 50 articles (API restriction)

TSLA:
  Price: 1444 days (2020-01-02 to 2025-09-30)
  News:  ❌ No news articles found

MSFT:
  Price: 1444 days (2020-01-02 to 2025-09-30)
  N

In [ ]:
# OPTIONAL: Force delete all old data and recollect
print("⚠️  WARNING: This will delete ALL existing price and news data!")
print("Only run this if you want to force a complete data refresh.\n")

user_confirm = input("Type 'YES' to confirm deletion: ")

if user_confirm == "YES":
    import os
    import glob
    
    # Delete all price and news data
    for symbol in SYMBOLS:
        price_path = f"data/price_data/{symbol}_price.json"
        news_path = f"data/news_data/{symbol}_news.json"
        if os.path.exists(price_path):
            os.remove(price_path)
            print(f"  Deleted: {price_path}")
        if os.path.exists(news_path):
            os.remove(news_path)
            print(f"  Deleted: {news_path}")
    
    print("\n✅ All data files deleted. Ready for fresh collection.")
    print("Run the main collection cell below to recollect data.")
else:
    print("\n❌ Deletion cancelled. Existing data preserved.")

Testing page_token pagination for AAPL in ALL of 2024

Iteration 1: Got 50 articles, 50 unique total → No more pages

✅ Total unique articles for AAPL in 2024: 50
Got 50 articles, 50 unique total → No more pages

✅ Total unique articles for AAPL in 2024: 50
